In [2]:
import pandas as pd

In [ ]:
# Сколько нужно выделить денег на рекламу в следующем месяце, чтобы увеличить выручку в 2 раза

# Сейчас выручка = х = 21K
# Нам в следующем месяце нужно 2х = 42K
# Итоговая выручка = 
    # выручка от новых клиентов = 27.3K
    # + выручка от текущих клиентов = 14.7K
# какую долю выручки можно ожидать от текущих клиентов?
# остаток - то, что нужно 'докупить' маркетингом 

In [ ]:
# 1. Построим календарный когортный отчет 
# 2. Считаем долю выручки от новых клиентов 
# ...

In [ ]:
# Алгоритм построения когортного отчета
# 0. Формулируем задачу 
# 1. Определяем когорту (Событие и временной промежуток) - Первая покупка по месяцам 
# 2. Выделяем целевую метрику - выручка
# 3. Выбираем подходящий формат отчета - календарный

In [14]:
orders = pd.read_csv('orders.csv', index_col='order_id')
orders['order_created_dt'] = pd.to_datetime(orders['order_created_dt'])
orders['payment_month'] = orders['order_created_dt'].dt.to_period('M')
orders = (
    orders
    .groupby('user_idi')
    .agg(first_payment_at = ('order_created_dt', 'min'))
    .merge(orders, how='inner', left_index=True, right_on='user_idi')
)
orders['payment_month'] = orders['order_created_dt'].dt.to_period('M')
orders['first_payment_month'] = orders['first_payment_at'].dt.to_period('M')
orders.head(4)


,first_payment_at,user_idi,order_amount,order_created_dt,payment_month,first_payment_month
order_id,,,,,,
906,2021-01-08,1443,6440,2021-01-08,2021-01,2021-01
843,2020-10-09,1446,6440,2020-10-09,2020-10,2020-10
995,2021-06-04,1451,6440,2021-06-04,2021-06,2021-06
1030,2021-11-23,1469,6440,2021-11-23,2021-11,2021-11


In [ ]:
# ( # Способ построения когортного отчета через группировку
#     orders
#     .groupby(['first_payment_month', 'payment_month'])
#     .agg(revenue = ('order_amount', 'sum'))
#     .tail(15)  # tail - это head наоборот, показывает с конца
# )

revenue
first_payment_month payment_month         
2021-02             2021-02          77280
                    2021-03          39800
2021-03             2021-03          77860
2021-04             2021-04         149860
                    2021-05          19900
                    2021-07           6440
2021-05             2021-05          97760
                    2021-06          19900
2021-06             2021-06          90740
2021-07             2021-07          71420
2021-08             2021-08          25760
2021-09             2021-09          25760
2021-10             2021-10          19320
2021-11             2021-11          19320
2021-12             2021-12          32780

In [29]:
# Способ построения когортного отчета через сводные таблицы
cohorts = (
    orders
    .pivot_table(
        index='first_payment_month',
        columns='payment_month',
        values='order_amount',
        aggfunc='sum'
    )
)
cohorts.tail(15)

payment_month,2020-02,2020-03,2020-04,2020-05,2020-06,2020-07,2020-08,2020-09,2020-10,2020-11,...,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12
first_payment_month,,,,,,,,,,,,,,,,,,,,,
2020-10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,208400.0,19900.0,...,NaN,NaN,NaN,NaN,NaN,NaN,6440.0,NaN,NaN,NaN
2020-11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,155720.0,...,NaN,NaN,NaN,NaN,6440.0,26340.0,NaN,NaN,NaN,NaN
2020-12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6440.0,NaN,6440.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,39800.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,77860.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,149860.0,19900.0,NaN,6440.0,NaN,NaN,NaN,NaN,NaN
2021-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,97760.0,19900.0,NaN,NaN,NaN,NaN,NaN,NaN
2021-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,90740.0,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
[cohorts.at[m,m] for m in cohorts.columns] / cohorts.sum(axis='rows')

payment_month
2020-02    1.000000
2020-03    1.000000
2020-04    1.000000
2020-05    1.000000
2020-06    0.693752
2020-07    1.000000
2020-08    0.531882
2020-09    1.000000
2020-10    1.000000
2020-11    0.855322
2020-12    0.809021
2021-01    0.662111
2021-02    0.437154
2021-03    0.596446
2021-04    0.958797
2021-05    0.787752
2021-06    0.820137
2021-07    0.847212
2021-08    0.494434
2021-09    0.800000
2021-10    1.000000
2021-11    1.000000
2021-12    1.000000
Freq: M, dtype: float64

In [ ]:
# cohorts.at['2020-11', '2020-11']
# [cohorts.at[m,m] for m in cohorts.columns]

[np.float64(45660.0),
 np.float64(19320.0),
 np.float64(6440.0),
 np.float64(25760.0),
 np.float64(45080.0),
 np.float64(84300.0),
 np.float64(97760.0),
 np.float64(116500.0),
 np.float64(208400.0),
 np.float64(155720.0),
 np.float64(168600.0),
 np.float64(168600.0),
 np.float64(77280.0),
 np.float64(77860.0),
 np.float64(149860.0),
 np.float64(97760.0),
 np.float64(90740.0),
 np.float64(71420.0),
 np.float64(25760.0),
 np.float64(25760.0),
 np.float64(19320.0),
 np.float64(19320.0),
 np.float64(32780.0)]